# EndurancePy — full quickstart

A guided tour of the package on real endurance data (Al Kamel archives).
**Requires network access.** Inspired by FastF1.

> No Al Kamel data is bundled with the project — please respect the portals' terms of service.

In [ ]:
from pathlib import Path

import endurancepy as ep

ep.set_log_level("INFO")  # surface discovery/download progress

Path("./cache").mkdir(exist_ok=True)
ep.Cache.enable_cache("./cache")

## 1. Season calendar

Season ids look like `08_2018-2019` / `13_2024` (see the portal URL).

In [ ]:
schedule = ep.get_event_schedule(2019, "WEC", season="08_2018-2019")
schedule[["RoundNumber", "EventName"]]

## 2. Load a session

The calendar lists only the events. An event's **date span** and **session
schedule** (start time + duration) live on its own page, so they're fetched on
demand with `event.get_dates()` / `event.get_sessions()`. Pick an event, then
load its race (the session already knows its season).

In [ ]:
event = schedule.get_event_by_name("Spa")

# dates and sessions live on the event's own page (fetched on demand)
print("Weekend:", event.get_dates())  # (first day, last day) — dates only, no time

# Session, StartTime (date + time of day) and Duration (race length where known)
sessions = event.get_sessions()
sessions.assign(Duration=sessions["Duration"].map(ep.format_timedelta))

In [ ]:
session = event.get_race()
session.load()
len(session.laps), len(session.cars)

## 3. Classification (overall & per class)

In [ ]:
# Positions and lap counts read as integers; format the lap time for display.
top = session.results[
    ["Position", "CarNumber", "Class", "Crew", "Laps", "BestLapTime"]
].head(10)
top.assign(BestLapTime=top["BestLapTime"].map(ep.format_laptime))

In [ ]:
session.results.pick_classes("LMGTE Am").head()

## 4. Laps and the `pick_*` filters

In [ ]:
laps = session.laps

for class_name in sorted(laps["Class"].dropna().unique()):
    best = laps.pick_classes(class_name).pick_fastest()
    if best is not None:
        time = ep.format_laptime(best["LapTime"])
        print(f"{class_name:10} {time}  (car {best['CarNumber']})")

In [ ]:
# Clean green-flag laps of the leading car, by stint
lead = session.cars[0]
clean = laps.pick_cars(lead).pick_wo_box()
green = clean.pick_track_status("GF")
# Older seasons (e.g. 2018-2019) have no per-lap flag, so green-flag filtering
# would drop every lap; fall back to all non-pit laps in that case.
car = green if len(green) else clean

agg = car.groupby("Stint")["LapTime"].agg(["count", "min"])
agg.assign(min=agg["min"].map(ep.format_laptime))

## 5. Track status & weather

In [ ]:
session.track_status

In [ ]:
session.weather_data.head()

## 6. Interactive charts (Plotly)

Endurance fields are large and races are long, so the chart helpers are
**interactive** (zoom into a window, hover for details, click the legend to
isolate a class). Needs the `interactive` extra: `pip install endurancepy[interactive]`.
Each helper returns a `plotly.graph_objects.Figure`.

In [ ]:
from endurancepy import plotting

# Stint strategy: one bar per stint, per car; the gaps are pit stops.
plotting.plot_strategy(session)

In [ ]:
# Lap-time evolution per car (y-axis reads M:SS.mmm), coloured by class.
plotting.plot_lap_evolution(session)

In [ ]:
# Race trace (delta to a reference pace) with FCY / safety-car windows shaded.
fig = plotting.plot_race_trace(session)
plotting.add_track_status(fig, session)
fig

## 7. Championship standings

Pass one `session.results` per round to `compute_standings` (configurable points / per class).

In [ ]:
# Illustrative: a single round here; in practice collect every round's results.
ep.compute_standings([session.results], by="CarNumber", per_class=True).head(10)